# Module 1: Interpretable Ensemble for Fake Job Posting Detection

**Team:** @tommygarner @ethandavenport @nkfavoriti @sebaspalacino

**Course:** Advanced Machine Learning (UT Austin)

## Features
- **Multi-level Interpretability:** Feature importance, sentence attribution, linguistic patterns
- **Three-tier Risk Classification:** HIGH/MEDIUM/LOW risk levels
- **Rule-Based Boosting:** 8 red flags detect fraud indicators
- **LIME for both NB and LSTM** interpretation
- **SHAP for NB and LSTM** (Sebastian's implementation)

In [ ]:
import os
from pathlib import Path

# 1. Clone your repo into /content if it's not already there
if not Path("/content/job-postings-fraud").exists():
    !git clone https://github.com/tommygarner/job-postings-fraud.git /content/job-postings-fraud

# 2. Set repo and models paths
REPO_ROOT = Path("/content/job-postings-fraud")
MODELS_DIR = REPO_ROOT / "models"

print("CWD:", os.getcwd())
print("Repo root:", REPO_ROOT)
print("Models dir exists:", MODELS_DIR.exists())
print("Model files:", [p.name for p in MODELS_DIR.iterdir()])

In [ ]:
# Install SHAP if needed
!pip install shap -q

In [ ]:
import kagglehub
import os
import pandas as pd

path = kagglehub.dataset_download("shivamb/real-or-fake-fake-jobposting-prediction")
print("Path to dataset files:", path)

filepath = os.path.join(path, "fake_job_postings.csv")
df = pd.read_csv(filepath)

In [ ]:
import nltk
nltk.download('punkt_tab')

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import joblib
import pickle
import torch
import scipy.sparse as sp
import shap

from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from lime.lime_text import LimeTextExplainer

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize, sent_tokenize

nltk.download("punkt")
nltk.download("wordnet")
nltk.download("stopwords")

# Paths
REPO_ROOT = Path("/content/job-postings-fraud")
MODELS_DIR = REPO_ROOT / "models"

# Prepare data
df["description"] = df["description"].fillna("")

# Text preprocessing to match NB training
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

def preprocess_text(s: str) -> str:
    tokens = word_tokenize(str(s).lower())
    tokens = [t for t in tokens if t.isalpha() and t not in stop_words]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return " ".join(tokens)

df["text_processed"] = df["description"].apply(preprocess_text)
print("✓ Data loaded and preprocessed")

In [ ]:
# Create combined text field for full analysis
def combine_text_fields(row):
    fields = ['title', 'company_profile', 'description', 'requirements', 'benefits']
    parts = [str(row.get(f, '') or '') for f in fields]
    return ' '.join(parts)

df['text_all'] = df.apply(combine_text_fields, axis=1)

def safe_display_field(val, max_len=80):
    s = str(val) if pd.notna(val) else "N/A"
    return s[:max_len] + "..." if len(s) > max_len else s

print("✓ Combined text field created")

In [ ]:
# === NB pipeline: TF-IDF + MultinomialNB ===
nb_pipeline = joblib.load(MODELS_DIR / "nb_pipeline.pkl")
vectorizer = nb_pipeline.named_steps["tfidfvectorizer"]
nb_model = nb_pipeline.named_steps["multinomialnb"]

def predict_nb_proba(text: str) -> np.ndarray:
    return nb_pipeline.predict_proba([preprocess_text(text)])[0]  # [p_legit, p_fraud]


# === LSTM + tokenizer ===
lstm_model = load_model(MODELS_DIR / "lstm_model.h5")
with open(MODELS_DIR / "tokenizer.pkl", "rb") as f:
    lstm_tokenizer = pickle.load(f)

MAX_LEN = 200

def predict_lstm_proba(text: str) -> np.ndarray:
    seq = lstm_tokenizer.texts_to_sequences([preprocess_text(text)])
    padded = pad_sequences(seq, maxlen=MAX_LEN, padding="post")
    p_fraud = float(lstm_model.predict(padded, verbose=0)[0, 0])
    return np.array([1 - p_fraud, p_fraud])


# === MiniLM (optional) ===
minilm_path = MODELS_DIR / "model_miniLM_final"
minilm_tokenizer = AutoTokenizer.from_pretrained(str(minilm_path))
minilm_model = AutoModelForSequenceClassification.from_pretrained(str(minilm_path))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
minilm_model.to(device).eval()

def predict_minilm_proba(text: str) -> np.ndarray:
    encoded = minilm_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=256,
        padding="max_length",
    ).to(device)
    with torch.no_grad():
        logits = minilm_model(**encoded).logits
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
    return probs

print("✓ NB pipeline, LSTM, MiniLM loaded")

## LIME Explainers Setup
LIME (Local Interpretable Model-agnostic Explanations) for both Naive Bayes and LSTM models.

In [ ]:
# === LIME for Naive Bayes ===
class_names = ["legit", "fraud"]
explainer_nb = LimeTextExplainer(class_names=class_names)

def nb_predict_for_lime(texts):
    return np.vstack([predict_nb_proba(t) for t in texts])

def explain_nb(text: str, num_features: int = 10):
    exp = explainer_nb.explain_instance(
        text_instance=text,
        classifier_fn=nb_predict_for_lime,
        num_features=num_features,
    )
    return exp

print("✓ LIME NB explainer initialized")

In [ ]:
# === LIME for LSTM ===
explainer_lstm = LimeTextExplainer(
    class_names=class_names,
    split_expression=r'\W+',
    bow=False,  # Important: preserve word order for LSTM
    random_state=42
)

def lstm_predict_for_lime(texts):
    """LIME wrapper: takes text list, returns [p_legit, p_fraud] array."""
    processed = [preprocess_text(t) for t in texts]
    seqs = lstm_tokenizer.texts_to_sequences(processed)
    padded = pad_sequences(seqs, maxlen=MAX_LEN, padding='post', truncating='post')
    fraud_probs = lstm_model.predict(padded, verbose=0).flatten()
    return np.column_stack([1 - fraud_probs, fraud_probs])

def explain_lstm(text: str, num_features: int = 15, num_samples: int = 1000):
    """Generate LIME explanation for LSTM."""
    return explainer_lstm.explain_instance(
        text_instance=text,
        classifier_fn=lstm_predict_for_lime,
        num_features=num_features,
        num_samples=num_samples,
        labels=(1,)  # Explain fraud class
    )

print("✓ LIME LSTM explainer initialized")

In [ ]:
# === LIME Explanation Functions ===

def lime_explain_lstm_for_text(text: str, top_k: int = 15, num_samples: int = 1000):
    """
    LIME explanation for LSTM - mirrors shap_explain_lstm_for_text interface.
    """
    exp = explain_lstm(text, num_features=top_k, num_samples=num_samples)
    word_weights = exp.as_list(label=1)
    tok_vals = [(word, score) for word, score in word_weights]
    return tok_vals, exp


def lime_explain_nb_for_text(text: str, top_k: int = 15):
    """
    LIME explanation for NB.
    """
    exp = explain_nb(text, num_features=top_k)
    word_weights = exp.as_list(label=1)
    tok_vals = [(word, score) for word, score in word_weights]
    return tok_vals, exp


def lime_sentence_attributions(text: str, top_k: int = 20, num_samples: int = 1000):
    """
    Roll up LIME word scores to sentence level.
    """
    tok_vals, _ = lime_explain_lstm_for_text(text, top_k=top_k, num_samples=num_samples)
    word_to_score = {word.lower(): score for word, score in tok_vals}
    
    sentences = sent_tokenize(text)
    sent_scores = []
    
    for sent in sentences:
        words = sent.lower().split()
        total = sum(word_to_score.get(w, 0) for w in words)
        sent_scores.append({
            "sentence": sent.strip(),
            "lime_sum": float(total),
            "lime_avg": float(total / max(len(words), 1))
        })
    
    sent_scores.sort(key=lambda d: -abs(d["lime_sum"]))
    return sent_scores


def explain_posting_with_lime(text: str, nb_top_k: int = 12, lstm_top_k: int = 15):
    """
    Full LIME explanation for both NB and LSTM.
    """
    nb_tok, nb_exp = lime_explain_nb_for_text(text, top_k=nb_top_k)
    lstm_tok, lstm_exp = lime_explain_lstm_for_text(text, top_k=lstm_top_k)
    sent_scores = lime_sentence_attributions(text, top_k=lstm_top_k)
    
    return {
        "nb": {"top_terms": nb_tok, "explanation": nb_exp},
        "lstm": {"token_attributions": lstm_tok, "explanation": lstm_exp,
                 "sentence_rollup": sent_scores[:5]}
    }

print("✓ LIME explanation functions defined")

## SHAP Explainers Setup
SHAP (SHapley Additive exPlanations) for both Naive Bayes and LSTM models.

In [ ]:
# === SHAP Helper Function ===

def _ensure_feature_dim(X_sparse, expected_dim):
    """
    Ensure sparse matrix has the expected number of features.
    Pads with zeros or truncates if necessary.
    """
    current_dim = X_sparse.shape[1]
    
    if current_dim == expected_dim:
        return X_sparse
    elif current_dim < expected_dim:
        # Pad with zeros
        padding = sp.csr_matrix((X_sparse.shape[0], expected_dim - current_dim))
        return sp.hstack([X_sparse, padding], format='csr')
    else:
        # Truncate
        return X_sparse[:, :expected_dim]

print("✓ SHAP helper functions defined")

In [ ]:
# === SHAP Explainers Setup ===

# 1) Build background text (small sample for speed)
BACKGROUND_N = 50
background_text = df["text_all"].dropna().sample(min(BACKGROUND_N, len(df)), random_state=7).tolist()

# 2) NB (TF-IDF) background matrix and dimension normalization
X_bg_sparse = vectorizer.transform(background_text)
EXPECTED = getattr(nb_model, "n_features_in_", X_bg_sparse.shape[1])
X_bg_sparse = _ensure_feature_dim(X_bg_sparse, EXPECTED)
X_bg_dense = X_bg_sparse[:50].toarray()  # KernelExplainer prefers dense background

# 3) NB predict wrapper returning positive class probability
def nb_predict_proba_pos(X_dense):
    X_csr = sp.csr_matrix(X_dense)
    exp = getattr(nb_model, "n_features_in_", X_csr.shape[1])
    X_csr = _ensure_feature_dim(X_csr, exp)
    return nb_model.predict_proba(X_csr)[:, 1]

# 4) NB SHAP explainer (Kernel)
nb_shap_explainer = shap.KernelExplainer(
    nb_predict_proba_pos,
    X_bg_dense
)

print("✓ SHAP NB explainer ready")
print("  NB expected features:", EXPECTED)

In [ ]:
# === SHAP LSTM Explainer ===

# Infer MAXLEN from model input
try:
    _shape = getattr(lstm_model, "input_shape", None)
    MAXLEN_SHAP = _shape[1] if (isinstance(_shape, (list, tuple)) and len(_shape) > 1 and _shape[1]) else 200
except Exception:
    MAXLEN_SHAP = 200

def lstm_predict_proba_pos(raw_texts):
    """Returns 1D: P(fraud) for each text."""
    seqs = lstm_tokenizer.texts_to_sequences(raw_texts)
    X = pad_sequences(seqs, maxlen=MAXLEN_SHAP, padding="post", truncating="post")
    preds = lstm_model.predict(X, verbose=0).flatten()
    return preds  # 1D array (n,)

# Text masker for SHAP
text_masker = shap.maskers.Text(tokenizer=None)

# LSTM SHAP explainer
lstm_shap_explainer = shap.Explainer(
    model=lstm_predict_proba_pos,
    masker=text_masker,
    algorithm="partition"
)

# Prime cache (optional)
try:
    _ = lstm_shap_explainer(background_text[:5])
    print("✓ SHAP LSTM explainer ready")
except Exception as e:
    print(f"⚠️ SHAP LSTM explainer warning: {e}")
    print("  LSTM SHAP may have issues - use LIME as fallback")

In [ ]:
# === SHAP Explanation Functions ===

def shap_explain_nb_for_text(text: str, top_k=15, nsamples=200):
    """Explain NB decision with KernelExplainer (positive class)."""
    # Vectorize, enforce dimension, and make dense
    X_sparse = vectorizer.transform([text])
    expected = getattr(nb_model, "n_features_in_", X_sparse.shape[1])
    X_sparse = _ensure_feature_dim(X_sparse, expected)
    X_dense = X_sparse.toarray()

    # SHAP values
    vals = nb_shap_explainer.shap_values(X_dense, nsamples=nsamples)
    if isinstance(vals, list):
        vals = vals[0]
    vals = np.array(vals).reshape(-1)  # (n_features,)

    # Feature names
    try:
        terms = np.array(vectorizer.get_feature_names_out())
    except Exception:
        terms = np.array([f"f{i}" for i in range(X_dense.shape[1])])

    if len(terms) != expected:
        if len(terms) < expected:
            terms = np.concatenate([terms, np.array([f"__pad_{i}" for i in range(expected - len(terms))])])
        else:
            terms = terms[:expected]

    # Top-k
    idx = np.argsort(-np.abs(vals))[:top_k]
    items = [(terms[i], float(vals[i])) for i in idx]

    # Build Explanation for waterfall plot
    exp = shap.Explanation(
        values=vals,
        base_values=nb_shap_explainer.expected_value,
        data=X_dense[0],
        feature_names=terms
    )
    return items, exp


def shap_explain_lstm_for_text(text: str, max_display=20):
    """Explain LSTM decision with SHAP text explainer."""
    exp = lstm_shap_explainer([text])
    toks = exp.data[0]
    vals = np.array(exp.values[0])
    if vals.ndim == 2:
        vals = vals[:, -1]
    tok_vals = [(t, float(v)) for t, v in zip(toks, vals)]
    return tok_vals, exp


def explain_posting_with_shap(text: str, nb_top_k=12, lstm_max_display=20):
    """Full SHAP explanation for both NB and LSTM."""
    nb_top, nb_exp = shap_explain_nb_for_text(text, top_k=nb_top_k)
    lstm_tok, lstm_exp = shap_explain_lstm_for_text(text, max_display=lstm_max_display)

    # Roll-up token SHAP to sentences
    sents = sent_tokenize(text)
    sent_scores = []
    for s in sents:
        score = sum(v for tok, v in lstm_tok if tok.strip() and tok in s)
        sent_scores.append({"sentence": s, "shap_sum": float(score)})
    sent_scores.sort(key=lambda d: -abs(d["shap_sum"]))

    return {
        "nb": {"top_terms": nb_top, "explanation": nb_exp},
        "lstm": {"token_attributions": lstm_tok, "explanation": lstm_exp,
                 "sentence_rollup": sent_scores[:5]}
    }

print("✓ SHAP explanation functions defined")

## InterpretableEnsemble Class
Combines NB and LSTM with rule-based boosting and multi-level interpretability.

In [ ]:
class InterpretableEnsemble:
    def __init__(self, nb_pipeline, lstm_model, tokenizer, training_df=None):
        self.nb_pipeline = nb_pipeline
        self.lstm = lstm_model
        self.tokenizer = tokenizer
        self.training_df = training_df

        self.vectorizer = nb_pipeline.named_steps["tfidfvectorizer"]
        self.nb = nb_pipeline.named_steps["multinomialnb"]

        self.lemmatizer = WordNetLemmatizer()
        self.stop_words = set(stopwords.words("english"))

        print("✓ InterpretableEnsemble initialized")

    def predict_with_explanation(
        self,
        text: str,
        telecommuting: int = 0,
        has_company_logo: int = 0,
        has_questions: int = 0,
        ensemble_strategy: str = "rule_boosted",
    ):
        text_processed = self._preprocess_text(text)
        nb_text_features = self.vectorizer.transform([text_processed])

        location_fraud_ratio = 0.05
        character_count = len(text)

        nb_prob = float(self.nb_pipeline.predict_proba([text_processed])[0, 1])

        seq = self.tokenizer.texts_to_sequences([text_processed])
        padded = pad_sequences(seq, maxlen=MAX_LEN, padding="post")
        lstm_prob = float(self.lstm.predict(padded, verbose=0)[0, 0])

        patterns = self._detect_patterns(text)

        numeric_features = {
            "telecommuting": telecommuting,
            "has_company_logo": has_company_logo,
            "has_questions": has_questions,
            "location_fraud_ratio": location_fraud_ratio,
            "character_count": character_count,
        }

        if ensemble_strategy == "simple":
            fraud_prob = (nb_prob + lstm_prob) / 2.0
            strategy_info = "Simple averaging (50/50)"
            boost_details = []
        elif ensemble_strategy == "weighted":
            fraud_prob = 0.35 * nb_prob + 0.65 * lstm_prob
            strategy_info = "Weighted averaging (NB=35%, LSTM=65%)"
            boost_details = []
        else:
            fraud_prob, strategy_info, boost_details = self._rule_boosted_ensemble(
                nb_prob, lstm_prob, numeric_features, patterns, text
            )

        risk_level, risk_icon, recommendation = self._get_risk_classification(fraud_prob)

        explanation = self._generate_explanation(
            text=text,
            text_processed=text_processed,
            nb_features=nb_text_features,
            fraud_prob=fraud_prob,
            numeric_features=numeric_features,
        )

        return {
            "fraud_probability": fraud_prob,
            "fraud_probability_pct": f"{fraud_prob*100:.1f}%",
            "risk_level": risk_level,
            "risk_icon": risk_icon,
            "recommendation": recommendation,
            "ensemble_decision": "FRAUD" if fraud_prob > 0.5 else "REAL",
            "ensemble_strategy": strategy_info,
            "boost_details": boost_details,
            "confidence": self._calculate_confidence(nb_prob, lstm_prob, fraud_prob),
            "individual_predictions": {
                "naive_bayes_score": f"{nb_prob:.4f}",
                "lstm_score": f"{lstm_prob:.4f}",
                "agreement": abs(nb_prob - lstm_prob) < 0.15,
                "disagreement_margin": f"{abs(nb_prob - lstm_prob):.4f}",
            },
            "explanation": explanation,
            "features_used": numeric_features,
        }

    def _preprocess_text(self, text):
        tokens = word_tokenize(str(text).lower())
        tokens = [t for t in tokens if t.isalpha() and t not in self.stop_words]
        tokens = [self.lemmatizer.lemmatize(t) for t in tokens]
        return " ".join(tokens)

    def _generate_explanation(self, text, text_processed, nb_features, fraud_prob, numeric_features):
        return {
            "top_fraud_indicators": self._extract_top_features(nb_features),
            "sentence_attributions": self._highlight_risky_sentences(text, text_processed, nb_features),
            "linguistic_flags": self._detect_patterns(text),
            "numeric_features_impact": self._explain_numeric_features(numeric_features),
            "confidence_reasoning": self._explain_confidence(fraud_prob),
        }

    def _extract_top_features(self, nb_features, top_n=10):
        feature_names = np.array(self.vectorizer.get_feature_names_out())
        feature_log_prob_fraud = self.nb.feature_log_prob_[1][:len(feature_names)]
        feature_log_prob_real = self.nb.feature_log_prob_[0][:len(feature_names)]

        tfidf_scores = nb_features.toarray()[0]
        fraud_weight = feature_log_prob_fraud - feature_log_prob_real
        contribution = tfidf_scores * fraud_weight

        top_indices = np.argsort(contribution)[-top_n:][::-1]

        top_features = {}
        for idx in top_indices:
            if contribution[idx] > 0:
                feature = feature_names[idx]
                top_features[feature] = {
                    "fraud_weight": float(fraud_weight[idx]),
                    "tfidf_score": float(tfidf_scores[idx]),
                    "contribution": float(contribution[idx]),
                }
        return top_features

    def _highlight_risky_sentences(self, text, text_processed, nb_features, top_n=5):
        sentences = sent_tokenize(text)
        sentence_scores = []

        feature_names = np.array(self.vectorizer.get_feature_names_out())
        feature_log_prob_fraud = self.nb.feature_log_prob_[1][:len(feature_names)]
        feature_log_prob_real = self.nb.feature_log_prob_[0][:len(feature_names)]
        fraud_weights = feature_log_prob_fraud - feature_log_prob_real

        for sentence in sentences:
            sent_processed = self._preprocess_text(sentence)
            sent_features = self.vectorizer.transform([sent_processed])

            tfidf_scores = sent_features.toarray()[0]
            sent_fraud_score = np.sum(tfidf_scores * fraud_weights)
            sent_fraud_score /= (len(sent_processed.split()) + 1)

            patterns = self._detect_patterns_in_sentence(sentence)
            pattern_risk = len(patterns) * 0.15

            total_risk = max(0, min(1, sent_fraud_score + pattern_risk))

            sentence_scores.append({
                "sentence": sentence.strip(),
                "fraud_risk": total_risk,
                "fraud_risk_pct": f"{total_risk*100:.0f}%",
                "contributing_features": self._get_sentence_features(sent_features),
                "flagged_patterns": patterns,
            })

        sentence_scores.sort(key=lambda x: x["fraud_risk"], reverse=True)
        return sentence_scores[:top_n]

    def _get_sentence_features(self, sent_features, top_n=3):
        feature_names = np.array(self.vectorizer.get_feature_names_out())
        tfidf_scores = sent_features.toarray()[0]
        top_indices = np.argsort(tfidf_scores)[-top_n:][::-1]

        features = {}
        for idx in top_indices:
            if tfidf_scores[idx] > 0:
                features[feature_names[idx]] = float(tfidf_scores[idx])
        return features

    def _detect_patterns_in_sentence(self, sentence):
        patterns = []
        sent_lower = sentence.lower()

        if any(k in sent_lower for k in ["urgent", "immediately", "asap", "deadline", "quick", "hurry"]):
            patterns.append("Urgency indicator")
        if any(k in sent_lower for k in ["payment", "fee", "upfront", "investment", "wire", "bitcoin"]):
            patterns.append("Payment request")
        if any(k in sent_lower for k in ["flexible", "negotiable", "not specified", "tbd"]):
            patterns.append("Vague language")
        if "  " in sentence or ".." in sentence or "!!" in sentence:
            patterns.append("Grammar issues")

        return patterns

    def _detect_patterns(self, text):
        text_lower = text.lower()
        text_len = len(text)
        avg_sentence_len = np.mean([len(s.split()) for s in sent_tokenize(text)]) if text else 0

        return {
            "urgency_indicators": {
                "present": any(kw in text_lower for kw in ["urgent", "asap", "immediately"]),
                "examples": [kw for kw in ["urgent", "asap", "immediately"] if kw in text_lower],
            },
            "payment_requests": {
                "present": any(kw in text_lower for kw in ["payment", "fee", "upfront"]),
                "examples": [kw for kw in ["payment", "fee", "upfront"] if kw in text_lower],
            },
            "vague_language": {
                "present": any(kw in text_lower for kw in ["flexible", "tbd", "negotiable"]),
                "examples": [kw for kw in ["flexible", "tbd", "negotiable"] if kw in text_lower],
            },
            "text_quality": {
                "word_count": len(text.split()),
                "character_count": text_len,
                "avg_sentence_length": round(avg_sentence_len, 2),
            },
        }

    def _explain_numeric_features(self, numeric_features):
        explanations = []

        if numeric_features["telecommuting"] == 1:
            explanations.append("✅ Job offers telecommuting (slightly lower fraud risk)")
        else:
            explanations.append("⚠️ No telecommuting mentioned")

        if numeric_features["has_company_logo"] == 1:
            explanations.append("✅ Job posting has company logo (lower fraud risk)")
        else:
            explanations.append("🚨 No company logo (common in fraud postings)")

        if numeric_features["has_questions"] == 1:
            explanations.append("✅ Job has screening questions (lower fraud risk)")
        else:
            explanations.append("⚠️ No screening questions")

        if numeric_features["character_count"] < 500:
            explanations.append("🚨 Very short job description (red flag)")
        elif numeric_features["character_count"] > 5000:
            explanations.append("⚠️ Unusually long description")
        else:
            explanations.append("✅ Normal description length")

        return {"features": numeric_features, "interpretations": explanations}

    def _explain_confidence(self, fraud_prob):
        if fraud_prob > 0.8:
            return {"confidence_level": "VERY HIGH", "fraud_probability": fraud_prob,
                    "reasoning": "Strong fraud indicators detected", "recommendation": "DO NOT APPLY"}
        elif fraud_prob > 0.6:
            return {"confidence_level": "HIGH", "fraud_probability": fraud_prob,
                    "reasoning": "Multiple fraud indicators present", "recommendation": "PROCEED WITH CAUTION"}
        elif fraud_prob > 0.4:
            return {"confidence_level": "MODERATE", "fraud_probability": fraud_prob,
                    "reasoning": "Mixed signals detected", "recommendation": "VERIFY company information"}
        elif fraud_prob > 0.2:
            return {"confidence_level": "LOW", "fraud_probability": fraud_prob,
                    "reasoning": "Mostly safe indicators", "recommendation": "LIKELY SAFE"}
        else:
            return {"confidence_level": "VERY LOW", "fraud_probability": fraud_prob,
                    "reasoning": "Strong legitimate indicators", "recommendation": "SAFE"}

    def _rule_boosted_ensemble(self, nb_prob, lstm_prob, numeric_features, patterns, text):
        base_prob = 0.25 * nb_prob + 0.75 * lstm_prob
        boost = 0.0
        boost_details = []

        if numeric_features["has_company_logo"] == 0:
            boost += 0.05
            boost_details.append("🚩 No company logo (+5%)")

        if numeric_features["has_questions"] == 0:
            boost += 0.05
            boost_details.append("🚩 No screening questions (+5%)")

        if patterns["urgency_indicators"]["present"]:
            keywords = ", ".join(patterns["urgency_indicators"]["examples"][:3])
            boost += 0.08
            boost_details.append(f"🚩 Urgency language: {keywords} (+8%)")

        if patterns["payment_requests"]["present"]:
            keywords = ", ".join(patterns["payment_requests"]["examples"][:3])
            boost += 0.12
            boost_details.append(f"🚩 Payment mentions: {keywords} (+12%)")

        if patterns["vague_language"]["present"]:
            keywords = ", ".join(patterns["vague_language"]["examples"][:2])
            boost += 0.04
            boost_details.append(f"🚩 Vague terms: {keywords} (+4%)")

        if numeric_features["character_count"] < 300:
            boost += 0.06
            boost_details.append(f"🚩 Very short description ({numeric_features['character_count']} chars) (+6%)")

        if lstm_prob > 0.35 and nb_prob < 0.05:
            boost += 0.10
            boost_details.append(f"🚩 LSTM pattern detection (LSTM={lstm_prob:.1%}, NB={nb_prob:.1%}) (+10%)")

        fraud_prob = min(base_prob + boost, 0.95)

        if boost > 0:
            strategy_desc = f"Rule-boosted ensemble (base: {base_prob:.1%} + boost: {boost:.1%} = {fraud_prob:.1%})"
        else:
            strategy_desc = "Weighted ensemble (no red flags detected)"

        return fraud_prob, strategy_desc, boost_details

    def _get_risk_classification(self, fraud_prob):
        if fraud_prob > 0.60:
            return ("HIGH RISK", "🔴", "DO NOT APPLY - Strong fraud indicators detected")
        elif fraud_prob > 0.20:
            return ("MEDIUM RISK", "🟡", "INVESTIGATE CAREFULLY - Verify company details")
        else:
            return ("LOW RISK", "🟢", "APPEARS SAFE - Standard precautions recommended")

    def _calculate_confidence(self, nb_prob, lstm_prob, fraud_prob):
        disagreement = abs(nb_prob - lstm_prob)
        if disagreement < 0.15:
            return "HIGH"
        elif fraud_prob > 0.80 or fraud_prob < 0.10:
            return "HIGH"
        elif disagreement < 0.30:
            return "MEDIUM"
        else:
            return "LOW"

print("✓ InterpretableEnsemble class defined")

In [ ]:
# Initialize the ensemble
ensemble = InterpretableEnsemble(
    nb_pipeline=nb_pipeline,
    lstm_model=lstm_model,
    tokenizer=lstm_tokenizer,
    training_df=df,
)

# Quick test
sample_text = df["description"].iloc[0]
result = ensemble.predict_with_explanation(sample_text)
print(f"Fraud probability: {result['fraud_probability_pct']}")
print(f"Risk level: {result['risk_level']}")

## Testing: Synthetic Fraud Job

In [ ]:
# === Test on Synthetic Fraud Job ===
synth_fraud = """
Work from home, no experience needed.
We will send you a check, you keep part of the money and send the rest back.
Provide your bank account details and a copy of your ID to get started immediately.
"""

print("=" * 60)
print("TEST: Synthetic Fraud Job")
print("=" * 60)

# Ensemble prediction
result = ensemble.predict_with_explanation(synth_fraud)
print(f"\n🔍 FRAUD PROBABILITY: {result['fraud_probability_pct']}")
print(f"{result['risk_icon']} RISK LEVEL: {result['risk_level']}")
print(f"📊 DECISION: {result['ensemble_decision']}")
print(f"✅ CONFIDENCE: {result['confidence']}")

print(f"\nNB Score: {result['individual_predictions']['naive_bayes_score']}")
print(f"LSTM Score: {result['individual_predictions']['lstm_score']}")

if result['boost_details']:
    print("\n=== Rule-based boosts ===")
    for d in result['boost_details']:
        print(f"  {d}")

In [ ]:
# === LIME Explanations ===
print("\n" + "=" * 60)
print("LIME EXPLANATIONS")
print("=" * 60)

print("\n== LIME: Naive Bayes top contributors ==")
nb_tok_vals, nb_exp = lime_explain_nb_for_text(synth_fraud, top_k=10)
for word, score in nb_tok_vals:
    arrow = "↑" if score > 0 else "↓"
    print(f"  {arrow} {word:20s} {score:+.4f}")

print("\n== LIME: LSTM token attributions ==")
lstm_tok_vals, lstm_lime_exp = lime_explain_lstm_for_text(synth_fraud, top_k=15)
for word, score in lstm_tok_vals:
    arrow = "↑" if score > 0 else "↓"
    print(f"  {arrow} {word:20s} {score:+.4f}")

In [ ]:
# LIME Visualization
print("📊 LIME Visualization: LSTM")
lstm_lime_exp.show_in_notebook(text=True)

In [ ]:
# === SHAP Explanations ===
print("\n" + "=" * 60)
print("SHAP EXPLANATIONS")
print("=" * 60)

print("\n== SHAP: Naive Bayes top contributors ==")
try:
    nb_shap_items, nb_shap_exp = shap_explain_nb_for_text(synth_fraud, top_k=10, nsamples=150)
    for term, val in nb_shap_items:
        arrow = "↑" if val > 0 else "↓"
        print(f"  {arrow} {term:20s} {val:+.4f}")
except Exception as e:
    print(f"  Error: {e}")

print("\n== SHAP: LSTM token attributions ==")
try:
    lstm_shap_tok, lstm_shap_exp = shap_explain_lstm_for_text(synth_fraud, max_display=15)
    for tok, v in lstm_shap_tok[:15]:
        arrow = "↑" if v > 0 else "↓"
        print(f"  {arrow} {tok!r:20s} {v:+.4f}")
except Exception as e:
    print(f"  Error: {e}")
    print("  Note: Use LIME for LSTM as fallback")

In [ ]:
# SHAP Visualizations
print("📊 SHAP Visualization: NB Waterfall")
try:
    shap.plots.waterfall(nb_shap_exp, max_display=15)
except Exception as e:
    try:
        shap.plots.waterfall(nb_shap_exp[0], max_display=15)
    except:
        print(f"  Could not display: {e}")

print("\n📊 SHAP Visualization: LSTM Text")
try:
    shap.plots.text(lstm_shap_exp[0])
except Exception as e:
    print(f"  Could not display: {e}")

## Testing: Real Dataset Samples

In [ ]:
# === Test on Real Fraudulent Job ===
print("=" * 80)
print("TEST: Real Fraudulent Job from Dataset")
print("=" * 80)

fraud_jobs = df[df['fraudulent'] == 1].copy()
fraud_sample = fraud_jobs.sample(1, random_state=42).iloc[0]
fraud_text = fraud_sample.get("text_all", fraud_sample.get("description", ""))

print("\nTitle:", safe_display_field(fraud_sample.get("title", "N/A")))
print("Location:", safe_display_field(fraud_sample.get("location", "N/A")))

fraud_result = ensemble.predict_with_explanation(fraud_text)
print(f"\n🔍 FRAUD PROBABILITY: {fraud_result['fraud_probability_pct']}")
print(f"{fraud_result['risk_icon']} RISK LEVEL: {fraud_result['risk_level']}")
print(f"📊 DECISION: {fraud_result['ensemble_decision']}")
print(f"✅ CONFIDENCE: {fraud_result['confidence']}")

# LIME
print("\n== LIME: NB top contributors ==")
nb_tok, _ = lime_explain_nb_for_text(fraud_text, top_k=10)
for word, score in nb_tok[:10]:
    arrow = "↑" if score > 0 else "↓"
    print(f"  {arrow} {word:20s} {score:+.4f}")

print("\n== LIME: LSTM token attributions ==")
lstm_tok, lstm_exp_fraud = lime_explain_lstm_for_text(fraud_text, top_k=15)
for word, score in lstm_tok[:15]:
    arrow = "↑" if score > 0 else "↓"
    print(f"  {arrow} {word:20s} {score:+.4f}")

# SHAP
print("\n== SHAP: NB top contributors ==")
try:
    nb_items, nb_exp = shap_explain_nb_for_text(fraud_text, top_k=12, nsamples=150)
    for term, val in nb_items:
        arrow = "↑" if val > 0 else "↓"
        print(f"  {arrow} {term:20s} {val:+.4f}")
except Exception as e:
    print(f"  Error: {e}")

print("\n== SHAP: LSTM tokens ==")
try:
    tok_vals, lstm_exp = shap_explain_lstm_for_text(fraud_text, max_display=20)
    for tok, v in tok_vals[:15]:
        arrow = "↑" if v > 0 else "↓"
        print(f"  {arrow} {tok!r:20s} {v:+.4f}")
except Exception as e:
    print(f"  Error: {e}")

In [ ]:
# Visualizations for real fraud
print("📊 LIME Visualization: Real Fraud Job")
lstm_exp_fraud.show_in_notebook(text=True)

In [ ]:
# === Test on Legitimate Job ===
print("\n" + "=" * 80)
print("TEST: Legitimate Job from Dataset")
print("=" * 80)

real_jobs = df[df['fraudulent'] == 0].copy()
real_sample = real_jobs.sample(1, random_state=99).iloc[0]
real_text = real_sample.get("text_all", real_sample.get("description", ""))

print("\nTitle:", safe_display_field(real_sample.get("title", "N/A")))
print("Location:", safe_display_field(real_sample.get("location", "N/A")))

real_result = ensemble.predict_with_explanation(real_text)
print(f"\n🔍 FRAUD PROBABILITY: {real_result['fraud_probability_pct']}")
print(f"{real_result['risk_icon']} RISK LEVEL: {real_result['risk_level']}")
print(f"📊 DECISION: {real_result['ensemble_decision']}")
print(f"✅ CONFIDENCE: {real_result['confidence']}")

# LIME - legitimacy indicators
print("\n== LIME: LSTM legitimacy indicators (negative = safe) ==")
lstm_tok, lstm_exp_legit = lime_explain_lstm_for_text(real_text, top_k=15)
neg_sorted = sorted(lstm_tok, key=lambda x: x[1])[:12]
for word, score in neg_sorted:
    arrow = "↑" if score > 0 else "↓"
    print(f"  {arrow} {word:20s} {score:+.4f}")

# SHAP - legitimacy indicators
print("\n== SHAP: NB legitimacy indicators ==")
try:
    nb_items_r, nb_exp_r = shap_explain_nb_for_text(real_text, top_k=12, nsamples=150)
    neg_sorted = sorted(nb_items_r, key=lambda kv: kv[1])[:12]
    for term, val in neg_sorted:
        arrow = "↑" if val > 0 else "↓"
        print(f"  {arrow} {term:20s} {val:+.4f}")
except Exception as e:
    print(f"  Error: {e}")

In [ ]:
# Visualization for legitimate job
print("📊 LIME Visualization: Legitimate Job")
lstm_exp_legit.show_in_notebook(text=True)

## Batch Evaluation

In [ ]:
# === 10-posting sanity check ===
fraud_samples = fraud_jobs.sample(5, random_state=123).copy()
real_samples = real_jobs.sample(5, random_state=456).copy()

results_list = []
print("\nEvaluating 10 job postings...\n")

for label, sample_df in [("FRAUD", fraud_samples), ("REAL", real_samples)]:
    for _, row in sample_df.iterrows():
        text = row.get("text_all", row.get("description", ""))
        result = ensemble.predict_with_explanation(text)
        results_list.append({
            "True Label": label,
            "Predicted": result['ensemble_decision'],
            "Risk": result['risk_level'],
            "NB Score": result['individual_predictions']['naive_bayes_score'],
            "LSTM Score": result['individual_predictions']['lstm_score'],
            "Correct": (label == result['ensemble_decision']),
            "Title": safe_display_field(row.get("title", "N/A"), max_len=40)
        })

results_df = pd.DataFrame(results_list)
display(results_df[["True Label", "Predicted", "Risk", "NB Score", "LSTM Score", "Correct", "Title"]])

accuracy = results_df['Correct'].mean()
print(f"\n📊 Accuracy on sample: {accuracy:.2%}")
print("\nBreakdown by risk level:")
print(results_df.groupby(["True Label", "Risk"])["Correct"].agg(["count", "mean"]))

In [ ]:
print("\n" + "=" * 60)
print("✅ Module 1: Interpretable Ensemble Complete")
print("=" * 60)
print("\nFeatures implemented:")
print("  ✓ NB + LSTM ensemble with rule-based boosting")
print("  ✓ LIME explanations for both NB and LSTM")
print("  ✓ SHAP explanations for both NB and LSTM")
print("  ✓ Sentence-level attribution")
print("  ✓ Three-tier risk classification")
print("  ✓ Red flag detection")